# Med-DDPM Training — Conditional MRI Synthesis

Train a **Denoising Diffusion Probabilistic Model (DDPM)** to synthesize brain MRI slices from tumor segmentation masks.

This notebook is designed for **Google Colab** with GPU acceleration.

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone or update the repository
import os

REPO_URL = "https://github.com/AmineAitLaamim/Mask-to-MRI.git"
PROJECT_DIR = "/content/Mask-to-MRI"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL}
    print("✅ Repository cloned.")
else:
    %cd $PROJECT_DIR
    !git pull
    print("✅ Repository updated.")

In [ ]:
# Install dependencies
%cd $PROJECT_DIR
!pip install -q torch torchvision albumentations tifffile scikit-image pyyaml tqdm matplotlib

In [ ]:
# ── Mount Drive (persists checkpoints across disconnects) ──
import shutil
from google.colab import drive

USE_DRIVE = True

if USE_DRIVE:
    drive.mount('/content/drive', force_remount=True)
    DRIVE_PROJECT_DIR = "/content/drive/MyDrive/mask-to-mri"
    os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)

    # Symlink outputs TO Drive
    os.makedirs(os.path.join(PROJECT_DIR, "outputs"), exist_ok=True)
    drive_outputs = os.path.join(DRIVE_PROJECT_DIR, "outputs")
    os.makedirs(drive_outputs, exist_ok=True)
    for d in ["checkpoints", "samples", "metrics"]:
        local_dir = os.path.join(PROJECT_DIR, "outputs", d)
        remote_dir = os.path.join(drive_outputs, d)
        os.makedirs(remote_dir, exist_ok=True)
        if os.path.islink(local_dir):
            os.remove(local_dir)
        elif os.path.exists(local_dir):
            shutil.rmtree(local_dir)
        os.symlink(remote_dir, local_dir)

    print(f"✅ Drive mounted. Outputs symlinked to {drive_outputs}")

## 2. Upload Dataset

In [ ]:
# ── Optimized Dataset Setup ──
import os, shutil

os.makedirs(os.path.join(PROJECT_DIR, "dataset"), exist_ok=True)

local_folder = os.path.join(PROJECT_DIR, "dataset", "lgg-mri-segmentation")
link_path = os.path.join(PROJECT_DIR, "dataset", "lgg-mri-segmentation")
zip_local = '/content/lgg-mri-segmentation.zip'

# Clean up old link/folder
if os.path.islink(link_path):
    os.remove(link_path)
elif os.path.isdir(link_path):
    shutil.rmtree(link_path)

# Extract only if local folder doesn't exist
if not os.path.exists(local_folder):
    # Find zip in Drive
    zip_drive = None
    for candidate in [
        '/content/drive/MyDrive/mask-to-mri/dataset/lgg-mri-segmentation.zip',
        '/content/drive/My Drive/mask-to-mri/dataset/lgg-mri-segmentation.zip',
        os.path.join(PROJECT_DIR, 'dataset', 'lgg-mri-segmentation.zip'),
    ]:
        if os.path.exists(candidate):
            zip_drive = candidate
            break

    if zip_drive:
        print(f"Found zip: {zip_drive}")
        if not os.path.exists(zip_local):
            print("Copying zip to local disk...")
            shutil.copy2(zip_drive, zip_local)
            print("Copy complete.")
        print("Unzipping to local disk...")
        !unzip -q -o "$zip_local" -d /content/
        print("Unzip complete.")
        # Move extracted folder to dataset/
        if os.path.exists('/content/lgg-mri-segmentation'):
            shutil.move('/content/lgg-mri-segmentation', local_folder)
            print("✅ Dataset extracted and moved.")
    else:
        print("⚠️  Zip not found. Upload to Drive at MyDrive/mask-to-mri/dataset/")
elif os.path.exists(local_folder):
    print("✅ Dataset already extracted.")

print(f"   Path: {PROJECT_DIR}/dataset/lgg-mri-segmentation/")

## 3. Import Modules & Load Config

In [ ]:
# ── Force clear any cached src modules ──
import sys
for _mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[_mod]

sys.path.insert(0, PROJECT_DIR)

import torch
import yaml
from pathlib import Path

from src.med_ddpm import (
    create_unet,
    DDPM,
    build_ddpm_dataloaders,
    ddpm_train,
    EMA,
    find_latest_ddpm_checkpoint,
    generate_from_masks,
)

config_path = Path(PROJECT_DIR) / "config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("✅ Config loaded:")
print(f"   Timesteps:      {config['med_ddpm']['timesteps']}")
print(f"   Epochs:         {config['med_ddpm']['epochs']}")
print(f"   Batch size:     {config['med_ddpm']['batch_size']}")
print(f"   LR:             {config['med_ddpm']['lr']}")
print(f"   EMA decay:      {config['med_ddpm']['ema_decay']}")
print(f"   Schedule:       {config['med_ddpm']['schedule']}")

## 4. Initialize Model & Data Loaders

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

# Fix seeds
import random
import numpy as np
seed = config["data"]["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
print(f"✅ Random seed fixed: {seed}")

In [ ]:
# Build DataLoaders
ddpm_cfg = config["med_ddpm"]
data_cfg = config["data"]

raw_dir = Path(PROJECT_DIR) / data_cfg["raw_dir"]
if not raw_dir.exists():
    raise FileNotFoundError(f"Dataset not found at {raw_dir}.")

loaders = build_ddpm_dataloaders(
    raw_dir=str(raw_dir),
    image_size=data_cfg["image_size"],
    batch_size=ddpm_cfg["batch_size"],
    num_workers=2,
    seed=data_cfg["seed"],
    balanced=data_cfg["balanced"],
    tumor_ratio=data_cfg["tumor_ratio"],
)

print(f"✅ DataLoaders created:")
print(f"   Train: {len(loaders['train'].dataset)} samples")
print(f"   Val:   {len(loaders['val'].dataset)} samples")
print(f"   Test:  {len(loaders['test'].dataset)} samples")

In [ ]:
# Verify a batch
mask_batch, mri_batch = next(iter(loaders["train"]))
print(f"✅ Batch shape: mask={mask_batch.shape}, mri={mri_batch.shape}")

In [ ]:
# Create U-Net model
model = create_unet(
    in_channels=config["model"]["output_channels"],
    cond_channels=config["model"]["input_channels"],
    num_filters=ddpm_cfg["num_filters"],
    time_emb_dim=ddpm_cfg["time_emb_dim"],
    norm=ddpm_cfg["norm"],
    attention_resolutions=ddpm_cfg.get("attention_resolutions", [16]),
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model created: {n_params:,} parameters ({n_params/1e6:.2f}M)")

In [ ]:
# Create DDPM wrapper
ddpm = DDPM(
    model=model,
    timesteps=ddpm_cfg["timesteps"],
    beta_start=ddpm_cfg["beta_start"],
    beta_end=ddpm_cfg["beta_end"],
    schedule=ddpm_cfg["schedule"],
    device=device,
).to(device)

print(f"✅ DDPM initialized: {ddpm.timesteps} timesteps, {ddpm_cfg['schedule']} schedule")

## 5. Check for Resumable Checkpoint

In [ ]:
checkpoint_dir = Path(PROJECT_DIR) / config["paths"]["checkpoints"]
latest_ckpt = find_latest_ddpm_checkpoint(str(checkpoint_dir))

# Set START_FRESH = True to restart training from epoch 0 (e.g., after architecture changes)
START_FRESH = False

if latest_ckpt and not START_FRESH:
    print(f"Resuming from checkpoint: {latest_ckpt}")
else:
    if START_FRESH:
        print("Fresh start requested — deleting old checkpoints.")
        !rm -f "$checkpoint_dir/checkpoint_ddpm_epoch_*.pt"
        !rm -f outputs/samples/ddpm_* outputs/metrics/ddpm_*
    print("No checkpoint found. Starting from scratch.")

resume_from = str(latest_ckpt) if (latest_ckpt and not START_FRESH) else None

## 6. Start Training

In [ ]:
history = ddpm_train(
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    model=model,
    ddpm=ddpm,
    config=config,
    device=device,
    checkpoint_dir=str(checkpoint_dir),
    samples_dir=str(Path(PROJECT_DIR) / config["paths"]["samples"]),
    metrics_dir=str(Path(PROJECT_DIR) / config["paths"]["metrics"]),
    resume_from=resume_from,
)

## 7. Plot Training Loss

In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
losses = [h["loss"] for h in history]

plt.figure(figsize=(10, 5))
plt.plot(epochs, losses, "b-", linewidth=2)
if "val_loss" in history[0]:
    val_losses = [h["val_loss"] for h in history]
    plt.plot(epochs, val_losses, "r-", linewidth=2, label="Val")
    plt.legend()
plt.xlabel("Epoch")
plt.ylabel("L1 Loss")
plt.title("DDPM Training Loss (L1)")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Generate Sample Images

In [ ]:
# Load EMA weights
ema = EMA(model)
if latest_ckpt:
    from src.med_ddpm import load_ddpm_checkpoint
    load_ddpm_checkpoint(latest_ckpt, model, ema=ema, device=device)
    ema.apply_shadow()
    print("✅ Loaded EMA weights")
model.eval()

In [ ]:
import numpy as np
from PIL import Image

mask_batch, real_batch = next(iter(loaders["val"]))
mask_batch = mask_batch.to(device)
real_batch = real_batch.to(device)

# Fast DDIM sampling (50 steps) — change to ddpm.sample() for full 1000 steps
from src.med_ddpm import DDIMSampler
ddim = DDIMSampler(ddpm, ddim_steps=50, eta=0.0)

with torch.no_grad():
    fake_batch = ddim.sample(mask_batch[:4])

rows = []
for i in range(4):
    mask_np = ((mask_batch[i, 0].cpu().numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    real_np = ((real_batch[i].cpu().permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    fake_np = ((fake_batch[i].cpu().permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    mask_3ch = np.stack([mask_np] * 3, axis=-1)
    rows.append(np.concatenate([mask_3ch, fake_np, real_np], axis=1))

grid = np.concatenate(rows, axis=0)
plt.figure(figsize=(15, 15))
plt.imshow(grid)
plt.axis("off")
plt.title("Mask | DDPM Generated | Real", fontsize=16)
plt.show()

## 9. Save to Drive (Optional)

In [ ]:
# Outputs are already symlinked to Drive — no action needed.
# Your checkpoints, samples, and metrics are persisted automatically.

## 10. Evaluate Quality (Optional)

In [ ]:
from src.pix2pix import compute_ssim_batch, compute_psnr_batch

ssim = compute_ssim_batch(fake_batch, real_batch[:4])
psnr = compute_psnr_batch(fake_batch, real_batch[:4])
print(f"✅ Evaluation (4 samples): SSIM={ssim:.4f}, PSNR={psnr:.2f} dB")